In [ ]:

import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
train["avg_monthly_spend"] = train["TotalCharges"] / (train["tenure"] + 1)
test["avg_monthly_spend"] = test["TotalCharges"] / (test["tenure"] + 1)

train["tenure_group"] = pd.cut(
    train["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)

test["tenure_group"] = pd.cut(
    test["tenure"],
    bins=[0,12,24,48,60,100],
    labels=["0-1yr","1-2yr","2-4yr","4-5yr","5+yr"]
)

service_cols = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

train["num_services"] = (train[service_cols] == "Yes").sum(axis=1)
test["num_services"] = (test[service_cols] == "Yes").sum(axis=1)


y = train[TARGET]
X = train.drop(columns=[TARGET])

cat_features = X.select_dtypes(
    include=["object","string","category"]
).columns.tolist()


def objective(trial):

    params = {

        "iterations": 3000,

        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),

        "depth": trial.suggest_int("depth", 6, 10),

        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),

        "bagging_temperature": trial.suggest_float(
            "bagging_temperature", 0, 1
        ),

        "random_strength": trial.suggest_float(
            "random_strength", 0, 1
        ),

        "loss_function": "Logloss",

        "eval_metric": "AUC",

        "task_type": "GPU",

        "devices": "0",

        "verbose": 0
    }

    skf = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    auc_scores = []

    for train_idx, val_idx in skf.split(X, y):

        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]

        model = CatBoostClassifier(**params)

        model.fit(
            X_train,
            y_train,
            eval_set=(X_val, y_val),
            cat_features=cat_features,
            early_stopping_rounds=100,
            verbose=False
        )

        preds = model.predict_proba(X_val)[:,1]

        y_val_binary = y_val.map({"No":0,"Yes":1})

        auc = roc_auc_score(y_val_binary, preds)

        auc_scores.append(auc)

    return np.mean(auc_scores)


study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=200)

print("\nBest AUC:", study.best_value)

print("\nBest parameters:")
print(study.best_params)

In [ ]:
best_params = study.best_params

final_model = CatBoostClassifier(

    iterations=5000,

    learning_rate=best_params["learning_rate"],
    depth=best_params["depth"],
    l2_leaf_reg=best_params["l2_leaf_reg"],
    bagging_temperature=best_params["bagging_temperature"],
    random_strength=best_params["random_strength"],

    loss_function="Logloss",
    eval_metric="AUC",

    task_type="GPU",
    devices="0",

    verbose=200
)

final_model.fit(
    X,
    y,
    cat_features=cat_features
)

pred_probs = final_model.predict_proba(test)[:,1]

submission = pd.DataFrame({
    "id": test_ids,
    "Churn": pred_probs
})

submission.to_csv("submission_catboost_optunaV1.csv", index=False)

print("\nSubmission file created: submission_catboost_optuna.csv")